<a href="https://colab.research.google.com/github/Nayan-Dhande/Fake-sms-Detection/blob/main/Spam%20sms%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:



import pandas as pd
import numpy as np
import re
import nltk
from google.colab import files

# -------------------------------
# 1. UPLOAD DATASET
# -------------------------------


df = pd.read_csv("/content/spam.csv", encoding="ISO-8859-1")

# Keep only important columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# -------------------------------
# 2. CLEANING + PREPROCESSING
# -------------------------------
df.drop_duplicates(inplace=True)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)      # remove punctuation
    text = re.sub(r"\d+", "", text)          # remove numbers
    text = " ".join([
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ])
    return text

df['cleaned'] = df['message'].apply(clean_text)

# -------------------------------
# 3. SPLIT + TF-IDF VECTORIZATION
# -------------------------------
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label_num'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # Unigrams + Bigrams
    min_df=2,              # ignore rare words
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# -------------------------------
# 4. TRAIN MODELS
# -------------------------------
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    C=2,
    class_weight='balanced',
    max_iter=3000,
    solver='liblinear'
)

lr_model.fit(X_train_tfidf, y_train)

# -------------------------------
# 5. EVALUATION
# -------------------------------
from sklearn.metrics import accuracy_score, classification_report

pred_lr = lr_model.predict(X_test_tfidf)

print("\n📈 Logistic Regression Accuracy:", accuracy_score(y_test, pred_lr))
print("\n📌 Classification Report:\n", classification_report(y_test, pred_lr))

# -------------------------------
# 6. USER INPUT PREDICTION
# -------------------------------
def predict_message():
    print("\nType your SMS message:")
    user_msg = input("> ")

    cleaned = clean_text(user_msg)
    vectorized = vectorizer.transform([cleaned])

    prediction = lr_model.predict(vectorized)[0]

    print("\n🔍 Prediction Result:")
    print("-----------------------------------")
    print(" MESSAGE:", user_msg)
    print(" RESULT :", "📛 SPAM" if prediction == 1 else "✅ NOT SPAM")
    print("-----------------------------------")

# Run prediction
predict_message()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



📈 Logistic Regression Accuracy: 0.9729206963249516

📌 Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98       889
           1       0.89      0.92      0.91       145

    accuracy                           0.97      1034
   macro avg       0.94      0.95      0.94      1034
weighted avg       0.97      0.97      0.97      1034


Type your SMS message:
> free 100000 dollars jst dial

🔍 Prediction Result:
-----------------------------------
 MESSAGE: free 100000 dollars jst dial
 RESULT : 📛 SPAM
-----------------------------------


# New section